### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:
### Reading all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recrusively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['sourceE_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e} ")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents 

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 4 PDF files to process

Processing: Infection Control and Management of Hazardous Materials for the Dental Team, (Mosby){Chris H. Miller}(Feb 11, 2022, Mosby){114460959} libgen.li.pdf
  Loaded 352 pages

Processing: Internal Medicine for Dental Treatments - Patients with Medical Diseases{Toshimi Chiba, Hiroyuki Yamada}(Apr 20, 2024, Springer){115322772} libgen.li.pdf
  Loaded 359 pages

Processing: Long-Term Outcomes of Dental Implants Placed in Fibula-free Flaps Used for Reconstruction of Maxillo-Mandibular Defects.pdf
  Loaded 12 pages

Processing: Structured-Visual Model for Dental Examination in Autism Spectrum Disorder Children_ Cooperation and Compliance.pdf
  Loaded 12 pages

Total documents loaded: 735


In [8]:
### Text splitting to get them into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)
chunks

Split 735 documents into 3740 chunks

Example chunk:
Content: INFECTION CONTROL
and Management
of Hazardous Materials
for the Dental Team
l]||[I
j
*
I I
Sr * V
n
Student Resources on Evolve
Access Code InsideEvolve
®...
Metadata: {'producer': 'iLovePDF', 'creator': 'Elsevier', 'creationdate': '2021-11-24T18:46:04+05:30', 'trapped': 'False', 'authoritativedomain': '[2]', 'elsevierwebpdfspecifications': '6.5', 'ebx_publisher': 'Elsevier Health Sciences', 'robots': 'noindex', 'crossmarkmajorversiondate': '2010-04-23', 'crossmarkdomainexclusive': 'true', 'doi': '10.1016/C2018-0-05600-5', 'title': 'Infection Control and Management of Hazardous Materials for the Dental Team', 'author': 'Chris Miller', 'subject': 'Infection Control and Management of Hazardous Materials for the Dental Team, Seventh Edition (2023) 1-336. 978-0-323-76404-9', 'moddate': '2023-10-14T07:19:46+00:00', 'source': '..\\data\\pdf_files\\Infection Control and Management of Hazardous Materials for the Dental Team, (Mosby)

[Document(metadata={'producer': 'iLovePDF', 'creator': 'Elsevier', 'creationdate': '2021-11-24T18:46:04+05:30', 'trapped': 'False', 'authoritativedomain': '[2]', 'elsevierwebpdfspecifications': '6.5', 'ebx_publisher': 'Elsevier Health Sciences', 'robots': 'noindex', 'crossmarkmajorversiondate': '2010-04-23', 'crossmarkdomainexclusive': 'true', 'doi': '10.1016/C2018-0-05600-5', 'title': 'Infection Control and Management of Hazardous Materials for the Dental Team', 'author': 'Chris Miller', 'subject': 'Infection Control and Management of Hazardous Materials for the Dental Team, Seventh Edition (2023) 1-336. 978-0-323-76404-9', 'moddate': '2023-10-14T07:19:46+00:00', 'source': '..\\data\\pdf_files\\Infection Control and Management of Hazardous Materials for the Dental Team, (Mosby){Chris H. Miller}(Feb 11, 2022, Mosby){114460959} libgen.li.pdf', 'total_pages': 352, 'page': 0, 'page_label': 'Cover', 'sourceE_file': 'Infection Control and Management of Hazardous Materials for the Dental Tea

In [10]:
### Embedding and VectorStoreDB

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-mpnet-base-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """ 
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

            Returns: numpy array of embeddings with shape (len(texts), embedding_dim)
            """
        if not self.model:
            raise ValueError("Model not loaded")
            
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 580.57it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 768


### VectoreStore

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None 
        self.collection = None 
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 3740


In [ ]:
chunks

In [14]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 3740 texts...


Batches: 100%|██████████| 117/117 [16:20<00:00,  8.38s/it]


Generated embeddings with shape: (3740, 768)
Adding 3740 documents to vector store...
Successfully added 3740 documents to vector store
Total documents in collection: 7480


### Retriever Pipeline From VectorStore

In [15]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [16]:
rag_retriever.retrieve("What is the role of microorganisms?")

Retrieving documents for query: 'What is the role of microorganisms?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.83it/s]

Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fb5d8c97_43',
  'content': '2\nScope of Microbiology and Infection Control\n1\nROLE OF MICROORGANISMS  \nIN INFECTION CONTROL\nMicrobiology is the study of small life forms, including bacteria, \narchaea, special fungi called molds and yeasts, protozoa, cer -\ntain algae, archaea, and viruses. Several subdisciplines within \nmicrobiology, such as bacteriology, mycology (study of fungi), \nprotozoology, and virology, concentrate on specific types of \nmicroorganisms or on the activities of selected microorganisms, \nsuch as those important in the fields of medical microbiology, \ndental microbiology, food microbiology, industrial microbiol -\nogy, and environmental (aquatic, soil, sewage, and space) mi -\ncrobiology. Close relationships also exist between microbiology \nand the fields of immunology (study of the immune system) and \nbiochemistry (the chemistry of life forms).\nThe field of infection control  (preventing microbial con -\ntamination and infection) is seated d

### RAG Pipeline- VectorDB To LLM Output Generation

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [19]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [20]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None



Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


In [21]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("What is the mandible?")



Retrieving documents for query: 'What is the mandible?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.31it/s]

Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)


[{'id': 'doc_be991fc1_3634',
  'content': 'A ‘|= produceaestheticallypleasingresults,thepros-yy ae PY a thesismusthaveasignificantlytallerstructuredue( & y — totheresultinglackofvertical"alveolar"height,\n2 Prey iy thiscanresultinhigherloadingforcesontheim-\\ y " plants.°°\nOneofthewaystopreventthisisbyplacing\n—_— the fibulasegment 5 to10mmabovethemandibular\n= ——x{ a inferiorborder.Anotherimportantconsiderationis\nSS a. thespaceneededtoplacetheimplants,takedentalmY impressions,allowmastication,and delivera pros-\nthesisaminimumof15mm ofmouthopeningisFig.5.Pedicledfibulaafterosteotomies,platingand neededattheincisaledgeoftheanteriorteeth.\nplacementofimplantswithscrew-retainedprosthesis.Thereshouldbe a 10to15mm spacebetween(DatafromSalamO.Salman,RuiP.Fernandes,Sundeep4.superioredgeofthefibulaandtheocclusalR.Rawal, Immediate Reconstruction and Dental Reha- a\nbilitationofSegmentalMandibularDefects:Descrip-49 oftheopposingdentitiontoenablethedevel-',
  'metadata': {'sourceE_file': 'Long

### Integration Vectordb Context pipeline With LLM output

In [22]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response= llm.invoke([prompt])
    return response.content

In [ ]:
answer=rag_simple("How does one reconstruct the mandible?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'How does one reconstruct the mandible??'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.89it/s]

Generated embeddings with shape: (1, 768)
Retrieved 3 documents (after filtering)


The mandible is reconstructed using a patient-specific technique involving virtual surgical planning. 

A customized cutting guide is created to depict tumor margins, and a patient-specific guide for fibula osteotomy is used. 

Segments of the osteotomized fibula are then positioned in relation to the natural mandible and dentition, and a reconstruction plate is used to secure the fibula graft in place.


### Enhanced RAG Pipeline Features

In [26]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("How does one reconstruct the mandible?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'How does one reconstruct the mandible?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.36it/s]


Generated embeddings with shape: (1, 768)
Retrieved 3 documents (after filtering)
Answer: According to the provided context, the mandible is reconstructed using a fibula graft. The process involves virtual surgical planning, where a 3D computer-generated model is created to display the mandibular mass and tumor margins. The fibula is then osteotomized and positioned in relation to the natural mandible and dentition, with the ideal position being in the middle where it becomes more quadrilateral in shape. The fibula segments should not be smaller than 2 cm, and the reconstruction plate is patient-specific to ensure proper positioning.
Sources: [{'source': '..\\data\\pdf_files\\Long-Term Outcomes of Dental Implants Placed in Fibula-free Flaps Used for Reconstruction of Maxillo-Mandibular Defects.pdf', 'page': 3, 'score': 0.3500369191169739, 'preview': "112 Michael et al\nA\nJ\ny 8Bry ? c’ SS < \\J (GEwo\nfi.\\Su gfAy ‘ ‘\nps \\ w\n4 le 'Y\n—A vlaew® y\ns lAyw ai 5Sug 2}\ndD _ E\n\\whe)2\

In [27]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("How does one reconstruct the mandible?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'How does one reconstruct the mandible?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.28it/s]

Generated embeddings with shape: (1, 768)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
112 Michael et al
A
J
y 8Bry ? c’ SS < \J (GEwo
fi.\Su gfAy ‘ ‘
ps \ w
4 le 'Y
—A vlaew® y
s lAyw ai 5Sug 2}
dD _ E
\whe)2
7y \ hs Ne A)ay ¢ \a 2 ee GS
Pet 4
- ileNEP |1 q


Wi
Fig. 3. Virtual surgical planning for post-tumor ablation mandibular reconstruction.(A) A 3D computer-
generated model displayingthe mandibular mass on the leftside of the face.(B) Mandible with a customized
cuttingguide and depictingtumor margins. (C)Patient-specificguide for fibulaosteotomy. (D)The reconstructed
mandible demonstrates how the fibulasegments and reconstructionplateshould be positioned inrelationto the
natural mandible and dentition.(£)The reconstructed mandible featuring the patient-specificreconstruction
plateand the fibulagraft.(Data from Salam O. Salman, Rui P.Fernandes, Sundeep R. Rawal, Immediate Recon-
structionand Dental Rehabilitationof Segmental Mandibular Defects:Descriptionof a Novel Technique, Journal

112 Michael et al
A
J
y 8Bry ? c’ SS < \J (GEwo
fi.\Su gfAy ‘ ‘
ps \ w
4 le 'Y
—A vlaew® y
s lAyw ai 5Sug 2}
dD _ E
\whe)2
7y \ hs Ne A)ay ¢ \a 2 ee GS
Pet 4
- ileNEP |1 q
Wi
Fig. 3. Virtual surgical planning for post-tumor ablation mandibular reconstruct